In [1]:
from precompute_s2_labels import generate_csv_with_precomputed_s2_labels, generate_csv_with_country_labels
from loader import load_dataset

DATASET_PATH = "I:/dataset_sharded_TEST"
OUT_PATH = "./s2_labels/s2_labels_level_6_TEST.csv"

dataset = load_dataset(DATASET_PATH, shuffle=False)

generate_csv_with_precomputed_s2_labels(dataset, LEVEL = 8, OUT_PATH = OUT_PATH)

#https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/

generate_csv_with_country_labels(
    dataset=dataset,
    GEOJSON_PATH="ne_10m_admin_0_countries.geojson",
    OUT_PATH="./country_labels.csv"
)

In [2]:
from precompute_s2_labels import load_s2_labels, lookup_s2_id, number_of_unique_s2_ids

table = load_s2_labels(OUT_PATH)

panoid = "gkSJpHv0vktJTh3NxfyYYw"

s2_id = lookup_s2_id(table, panoid)

print(f"S2 ID: {s2_id}")

number_of_unique_class_labels = number_of_unique_s2_ids(table)

print(f"Number of unique S2 IDs: {number_of_unique_class_labels}")

S2 ID: 5069487475861225472
Number of unique S2 IDs: 503


In [3]:
from precompute_s2_labels import load_country_labels, lookup_country

country_table = load_country_labels("country_labels.csv")

country = lookup_country(country_table, panoid)

print(country)

Sweden


In [4]:
from s2sphere import Cell, CellId, LatLng

cell_id = CellId(s2_id)
cell = Cell(cell_id)

print("Level:", cell_id.level())

center = cell_id.to_lat_lng() #Center
print("Center lat/lon:", center.lat().degrees, center.lng().degrees)

Level: 8
Center lat/lon: 57.78174646686309 13.986332799457246


In [5]:
vertices = []
for i in range(4):
    vertex = cell.get_vertex(i)
    ll = LatLng.from_point(vertex)
    vertices.append((ll.lat().degrees, ll.lng().degrees))

import folium

m = folium.Map(location=[center.lat().degrees, center.lng().degrees], zoom_start=10)
folium.Polygon(vertices, color="blue", weight=2, fill=False).add_to(m)
m.save("s2_cell.html")

In [ ]:
import csv
def dedup_lookup_csv(IN_PATH: str, OUT_PATH: str):
    with open(IN_PATH, newline="") as fin, open(OUT_PATH, "w", newline="") as fout:
        r, w = csv.reader(fin), csv.writer(fout)
        header = next(r)
        w.writerow(header)
        seen = set()
        for row in r:
            tup = tuple(row)
            if tup in seen:
                continue
            seen.add(tup)
            w.writerow(row)

IN = "country_labels.csv"
OUT = "country_labels2.csv"

dedup_lookup_csv(IN,OUT)